# Digit recognition with *Convolution Neural Network*
We shall use:
- *Keras* *MNIST* data for digit recognition : https://keras.io/api/datasets/mnist/
- *Tensorflow*  https://www.tensorflow.org/  
- *keras* https://www.tensorflow.org/guide/keras

Dataset consists of $28\times 28$ grayscale images labeled with digits (0 through 9)

Make sure that you change *Runtime* to ***GPU***.

## Import

In [ ]:
import numpy as np # for computation
import pandas as pd # for data handling and analysis
import matplotlib.pyplot as plt # for plotting

# metrics to evaluate models
from sklearn.metrics import classification_report, confusion_matrix

# import keras for Convolution Neural Network
import keras
from keras import datasets, layers, models, optimizers, utils

## Read data
We shall use the MNIST digits classification dataset using the *mnist.load_data()* utility (see https://keras.io/api/datasets/mnist/) into the following *Numpy* arrays:
- Training: input *x_train*, output *y_train*
- Test: input *x_test*, output *y_test*

In [ ]:
%%time
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
print(f'{len(y_train)} training and {len(y_test)} test images.')
print("Shape of arrays:")
print(f"\tTraining -> x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"\tTesting -> x_test: {x_test.shape}, y_test: {y_test.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
60000 training and 10000 test images.
Shape of arrays:
	Training -> x_train: (60000, 28, 28), y_train: (60000,)
	Training -> x_train: (10000, 28, 28), y_train: (10000,)
CPU times: user 282 ms, sys: 45.4 ms, total: 328 ms
Wall time: 405 ms


## Display digits

In [ ]:
def displayImages(images, labels, nCols=10):
    """Displays images with labels (nCols per row)
    - images: list of vectors with 784 (28x28) grayscale values
    - labels: list of labels for images"""
    nRows = np.ceil(len(labels)/nCols).astype('int') # number of rows
    plt.figure(figsize=(2*nCols,2*nRows)) # figure size
    for i in range(len(labels)):
        plt.subplot(nRows,nCols,i+1)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)
        plt.imshow(images[i], interpolation='spline16', cmap='gray_r')
        plt.xlabel(f'{labels[i]}', fontsize=24)
    plt.tight_layout()
    plt.show()
    return

Let's display the first 50 training images with labels

In [ ]:
SHOW_IMAGES = False # set to True to display examples
if SHOW_IMAGES:
    images = [x_train[i] for i in range(50)] # pixel vectors for examples
    labels = [y_train[i] for i in range(50)] # labeled digits for examples
    displayImages(images, labels, 10)

## Function to transform inputs
Images in convolution networks are represented by 3D arrays of dimensions $(width, height, depth)$, where $depth$ refers to the number of channels in the image. An *RGB* image has 3 channels: Red, Green, and Blue. A grayscale image has a single channel. We shall define a function to convert the 2D images into 3D tensors with the required shape. We shall also normalize the data by mapping gray scale values (0-255) to a number between 0 and 1.

In [ ]:
%%time
def Xform(x): # number of channels in a grayscale image = 1
    return x.reshape(-1, 28, 28, 1)/255.0

print(f'Downloaded x_test.shape: {x_test.shape}')
print(f'Transformed x_test.shape: {Xform(x_test).shape}')

Downloaded x_test.shape: (10000, 28, 28)
Transformed x_test.shape: (10000, 28, 28, 1)
CPU times: user 10.5 ms, sys: 13.9 ms, total: 24.5 ms
Wall time: 22.9 ms


## Create model




We shall create a convolution neural network with 4 convolution layers followed by a densely connected layer and an output layer (with as many neurons as there are output classes).

In this example I use $(3 \times 3)$ filters, with the 4 convolution layers containg 16, 32, 64 and 128 filters, respectively.

After every 2 convolution layers, we shall use $(2 \times 2)$ max-pooling, followed by BatchNormalization and Dropout to reduce over-fitting.

You can experiment with different architechtures and parameter values.


In [ ]:
# create model
model = keras.Sequential()

# specify shape of input image
model.add(keras.Input(shape=(28, 28, 1)))

# Add first convolution layer with 16 filters
model.add(layers.Conv2D(16, (3,3), activation='relu', padding='same', name="Conv1"))

# Add second convolution layer with 32 filters
model.add(layers.Conv2D(32, (3,3), activation='relu', padding='same', name="Conv2"))

# Add pooling layer
model.add(layers.MaxPooling2D((2,2)))

# Regularize
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.3))

# Add third convolution layer with 64 filters
model.add(layers.Conv2D(64, (3,3), activation='relu', padding='same', name="Conv3"))

# Add fourth convolution layer with 128 filters
model.add(layers.Conv2D(128, (3,3), activation='relu', padding='same', name="Conv4"))

# Add pooling layer
model.add(layers.MaxPooling2D((2,2)))

# Regularize
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.3))

# Flatten the outputs from the last convolution layer
model.add(layers.Flatten()) # flatten to vector

# Add dense layer with 256 neurons
model.add(layers.Dense(256, activation='relu'))

# Regularize
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.3))

# Add output layer with softmax activation
model.add(layers.Dense(10, activation='softmax'))

# compile model
opt = optimizers.SGD(learning_rate=0.01, momentum=0.9) # optimizer used
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ Conv1 (Conv2D)                       │ (None, 28, 28, 16)          │             160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Conv2 (Conv2D)                       │ (None, 28, 28, 32)          │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 14, 14, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 14, 14, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Conv3 (Conv2D)                       │ (None, 14, 14, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ Conv4 (Conv2D)                       │ (None, 14, 14, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 7, 7, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 7, 7, 128)           │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 7, 7, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 6272)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 256)                 │       1,605,888 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_5 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,707,274 (6.51 MB)

 Trainable params: 1,706,442 (6.51 MB)

 Non-trainable params: 832 (3.25 KB)

None


## Train model

We shall train the model over a number of epochs and in batches of specified sizes. A subset of the training examples will be held back for validation.  

In [ ]:
%%time
epochs = 10 # number of training epochs
batch_size = 512 # batch_size for training

history = model.fit(Xform(x_train), utils.to_categorical(y_train),
                    epochs=epochs, batch_size=batch_size, validation_split=0.1)

Epoch 1/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 20s 103ms/step - accuracy: 0.7925 - loss: 0.7045 - val_accuracy: 0.1050 - val_loss: 4.0098
Epoch 2/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.9716 - loss: 0.0894 - val_accuracy: 0.1050 - val_loss: 5.4231
Epoch 3/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9783 - loss: 0.0690 - val_accuracy: 0.1438 - val_loss: 4.4972
Epoch 4/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9837 - loss: 0.0533 - val_accuracy: 0.5072 - val_loss: 1.5316
Epoch 5/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.9852 - loss: 0.0478 - val_accuracy: 0.9583 - val_loss: 0.1483
Epoch 6/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.9870 - loss: 0.0422 - val_accuracy: 0.9783 - val_loss: 0.0770
Epoch 7/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.9877 - loss: 0.0408 - val_accuracy: 0.9908 - val_loss: 0.0330
Epoch 8/10
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9890 - loss: 0.0343 - val_ac

## Plot history
Check for over-fitting

In [ ]:
SHOW_HISTORY = False # set to True to display history
if SHOW_HISTORY:
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'validation'], loc='upper left')
    plt.show()

## Evaluate model

### Predict classes and probabilities for test examples

In [ ]:
%%time
pred_prob = model.predict(Xform(x_test)) # predicted probabilities
pred_test = pred_prob.argmax(axis=1) # predicted labels (most likely label)
print("Predicted probabilities: %d rows and %d columns" %pred_prob.shape)
print("Number of examples predicted: %d" %len(pred_test))

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
Predicted probabilities: 10000 rows and 10 columns
Number of examples predicted: 10000
CPU times: user 2.08 s, sys: 155 ms, total: 2.23 s
Wall time: 2.69 s


Check what the predicted probabilities and predicted classes look like

In [ ]:
n_ex = 7 # show the first n_ex examples
print('Probabilities,', 'Predicted class')
for i in range(n_ex):
    print(pred_prob[i].round(2), pred_test[i])

Probabilities, Predicted class
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0.] 7
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0.] 2
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.] 1
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.] 0
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.] 4
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.] 1
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.] 4


### Classification report

In [ ]:
print(classification_report(y_test, pred_test, digits=4))

              precision    recall  f1-score   support

           0     0.9919    0.9980    0.9949       980
           1     0.9921    0.9991    0.9956      1135
           2     0.9961    0.9903    0.9932      1032
           3     0.9941    0.9950    0.9946      1010
           4     0.9969    0.9888    0.9928       982
           5     0.9933    0.9933    0.9933       892
           6     0.9958    0.9906    0.9932       958
           7     0.9932    0.9903    0.9917      1028
           8     0.9928    0.9938    0.9933       974
           9     0.9872    0.9931    0.9901      1009

    accuracy                         0.9933     10000
   macro avg     0.9933    0.9932    0.9933     10000
weighted avg     0.9933    0.9933    0.9933     10000



### Confusion matrix

In [ ]:
pd.DataFrame(confusion_matrix(y_test, pred_test))

,0,1,2,3,4,5,6,7,8,9
0,978,0,0,0,0,0,0,1,1,0
1,0,1134,0,0,0,0,1,0,0,0
2,1,2,1022,1,1,0,0,4,1,0
3,0,0,1,1005,0,3,0,0,1,0
4,0,0,0,0,971,0,1,0,1,9
5,1,0,0,3,0,886,2,0,0,0
6,3,2,0,0,1,2,949,0,1,0
7,0,3,3,1,0,0,0,1018,1,2
8,2,1,0,1,0,0,0,0,968,2
9,1,1,0,0,1,1,0,2,1,1002


## Other stuff

Display misclassified digits

In [ ]:
misclassified = [i for i in range(len(y_test)) if y_test[i] != pred_test[i]] # indices of misclassified test examples
images = [x_test[i] for i in misclassified] # pixel vectors for examples
labels = [(y_test[i], pred_test[i], max(pred_prob[i]).round(2)) for i in misclassified] # labeled digits for examples
nE = len(misclassified) # number of errors
print('%d (%4.2f%%)images misclassifieded' %(nE, 100*nE/len(y_test)))
print("Labels displayed as '(true, predicted, confidence)'")
SHOW_IMAGES = False # set to True to display examples
if SHOW_IMAGES:
    displayImages(images, labels, 10)

67 (0.67%)images misclassifieded
Labels displayed as '(true, predicted, confidence)'


Use trained model to recognize your own handwitten digits

In [ ]:
# https://gist.github.com/korakot/8409b3feec20f159d8a50b0a811d3bca

from IPython.display import HTML, Image
from google.colab.output import eval_js
from base64 import b64decode
import PIL

canvas_html = """
<canvas width=%d height=%d style="border:1px solid #000000;"></canvas>
<button>Predict digit</button>
<script>
var canvas = document.querySelector('canvas')
var ctx = canvas.getContext('2d')
ctx.lineWidth = %d
var button = document.querySelector('button')
var mouse = {x: 0, y: 0}
canvas.addEventListener('mousemove', function(e) {
  mouse.x = e.pageX - this.offsetLeft
  mouse.y = e.pageY - this.offsetTop
})
canvas.onmousedown = ()=>{
  ctx.beginPath()
  ctx.moveTo(mouse.x, mouse.y)
  canvas.addEventListener('mousemove', onPaint)
}
canvas.onmouseup = ()=>{
  canvas.removeEventListener('mousemove', onPaint)
}
var onPaint = ()=>{
  ctx.lineTo(mouse.x, mouse.y)
  ctx.stroke()
}
var data = new Promise(resolve=>{
  button.onclick = ()=>{
    resolve(canvas.toDataURL('image/png'))
  }
})
</script>
"""
def predictImage(infile, model):
    img = PIL.Image.open(infile).resize((20,20))
    img = np.array(img).mean(axis=2)
    m = img.max()
    img = img*255/m
    a = np.zeros((28,28))
    a[4:4+20,4:4+20] = img
    x = np.array([a]).reshape((1,28,28,1))/255.0
    p = model.predict(x)
    conf = p.max()
    digit = p.argmax(axis=1)[0]
    print("Predicted digit: %d, (confidence = %4.3f)"  %(digit, conf))
    return

def draw(filename='digit.png', w=200, h=200, line_width=20):
    print("Draw a digit in the box below and click the 'Predict digit' button")
    display(HTML(canvas_html % (w, h, line_width)))
    data = eval_js("data")
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return predictImage(filename, model)

Run the next cell, draw a digit, and click on "Predict digit"

In [ ]:
DRAW_DIGIT = True # set to True to draw a digit
if DRAW_DIGIT:
    draw()

Draw a digit in the box below and click the 'Predict digit' button


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 537ms/step
Predicted digit: 2, (confidence = 1.000)
